# OpenKind — Phase 4E-A2 QASPER Audit Repair

**Purpose:** repair and rerun the blinded QASPER applicability audit after Phase 4E-A exported an empty `state_text` for all 150 review rows. This notebook performs no training, needs no GPU, and never opens final.

**Workbench:** `4e.1.0`  
**Immutable corpus run:** `20260921T013558Z`  
**Diagnostic parent:** Phase 4D epoch 0 / retained Phase 4B.2.1 stratified checkpoint  
**Default mode:** `prepare_audit`  
**Runtime:** CPU only

## Why this repair exists

The first Phase 4E audit completed mechanically but was non-dispositive:

| Result | Observed value | Interpretation |
|---|---:|---|
| Review rows | 150 | Expected population was preserved. |
| Empty `state_text` | 150 / 150 | Critical audit-pack defect. |
| Ambiguous | 79 / 150 (`52.67%`) | Review coverage was only `47.33%`. |
| Decided agreement | 58 / 71 (`81.69%`) | Conditional on decided rows only. |
| Semantic-none-side ambiguity | 56 / 69 (`81.16%`) | The primary research question remained unresolved. |
| Answerable-side decided agreement | 58 / 58 | No obvious answerable-label corruption, but visible gold evidence could bias review. |
| Semantic-none-side decided agreement | 0 / 13 | Could not distinguish label error from missing state context. |

The precise implementation defect was a silent fallback:

```python
state_row.get("effective_state_text", "")
```

The frozen `states.parquet` schema contains the canonical field `text`, not `effective_state_text`. Pandas therefore returned `""` instead of raising. Phase 4E-A2 uses `state_row["text"]`, verifies the corresponding `text_sha256`, requires complete non-empty state coverage, and excludes gold evidence from the primary blinded review.


## Recommended run sequence

1. Leave `MODE = "prepare_audit"` and run top to bottom. Existing prepared artifacts are verified rather than overwritten.
2. Review `AUDIT_GUIDE.md` and complete `AUDIT_REVIEW_BLINDED.csv` as `AUDIT_REVIEW_COMPLETED.csv` in the same run directory. Judge only from the paper state, question, and option description.
3. Change only `MODE` to `"analyze_review"` and rerun.
4. Resolve rows emitted to `AUDIT_ADJUDICATION_PACKET.csv`; only then choose data repair or one preregistered representation-aware seed-17 experiment.

Do not change the parent, frozen threshold, seed, sample counts, taxonomy, or evidence-blinding policy inside this run label. Any such change requires a new run label and contract. Final remains closed; seeds 42/123 are not started here.


## 1. Runtime and dependency contract


In [1]:
import sys, subprocess, importlib.metadata as im, json, platform

def installed(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

required = {
    "numpy": "numpy>=2,<3",
    "pandas": "pandas>=2,<3",
    "pyarrow": "pyarrow>=18,<30",
    "scikit-learn": "scikit-learn>=1.6,<2",
    "matplotlib": "matplotlib>=3.9,<4",
}
missing = [spec for name, spec in required.items() if installed(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {name: installed(name) for name in required},
}
print(json.dumps(ENVIRONMENT, indent=2, allow_nan=False))


{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "packages": {
    "numpy": "2.1.3",
    "pandas": "2.2.3",
    "pyarrow": "23.0.1",
    "scikit-learn": "1.6.1",
    "matplotlib": "3.10.0"
  }
}


## 2. Mount Drive or select a local mirror


In [ ]:
import os
from pathlib import Path

DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
LOCAL_OVERRIDE = os.environ.get("OPENKIND_COLAB_ROOT")

if LOCAL_OVERRIDE:
    COLAB_ROOT = Path(LOCAL_OVERRIDE).expanduser().resolve()
    print("Using local OpenKind artifact mirror:", COLAB_ROOT)
else:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError(
            "Outside Colab, set OPENKIND_COLAB_ROOT to a local mirror of "
            "the Colab Notebooks artifact tree."
        ) from exc
    drive.mount("/content/drive")
    COLAB_ROOT = DEFAULT_COLAB_ROOT
    print("Using mounted Google Drive:", COLAB_ROOT)

if not COLAB_ROOT.is_dir():
    raise FileNotFoundError(f"Artifact root does not exist: {COLAB_ROOT}")


## 3. Configure the immutable repair audit


In [ ]:
import hashlib
import math
import re
from datetime import datetime, timezone

WORKBENCH_VERSION = "4e.1.0"
BASE_RUN_ID = "20260921T013558Z"
SEED = 17
RUN_LABEL = "qasper_error_audit_repair_s17"

# Change only this field between the two prescribed passes.
MODE = os.environ.get("OPENKIND_PHASE4E_A2_MODE", "prepare_audit")

if MODE not in {"prepare_audit", "analyze_review"}:
    raise ValueError("MODE must be prepare_audit or analyze_review.")
if not re.fullmatch(r"[a-z0-9][a-z0-9._-]{0,79}", RUN_LABEL):
    raise ValueError("RUN_LABEL must be a short path-safe identifier.")

CFG = {
    "schema": "openkind-phase4e-a2-qasper-audit-repair-config/v1",
    "workbench_version": WORKBENCH_VERSION,
    "base_run_id": BASE_RUN_ID,
    "seed": SEED,
    "run_label": RUN_LABEL,
    "mode": MODE,
    "source": "qasper",
    "state_text_column": "text",
    "primary_review_exposes_gold_evidence": False,
    "policy_development": {
        "include_all_false_negatives": True,
        "include_all_false_positives": True,
        "true_positive_controls": 10,
        "true_negative_controls": 10,
    },
    "calibration_gate": {
        "sample_false_negatives": 30,
        "sample_false_positives": 30,
        "sampling": "deterministic_state_round_robin_sha256",
    },
    "review_target_kinds": ["answerable", "semantic_none", "ambiguous"],
    "final_split_available": False,
    "final_opened": False,
}

BASE_RUN = COLAB_ROOT / "OpenDecision_Phase4A_StateQuery_results" / BASE_RUN_ID
PHASE4D_RUN = (
    COLAB_ROOT
    / "OpenDecision_Phase4D_EvidenceAware_Applicability_results"
    / BASE_RUN_ID
    / "b2_evidence_residual_s17"
)
PREVIOUS_AUDIT_DIR = (
    COLAB_ROOT
    / "OpenKind_Phase4E_QASPER_Audit_results"
    / BASE_RUN_ID
    / "qasper_error_audit_s17"
)
OUTPUT_ROOT = (
    COLAB_ROOT
    / "OpenKind_Phase4E_A2_QASPER_Audit_Repair_results"
    / BASE_RUN_ID
)
AUDIT_DIR = OUTPUT_ROOT / RUN_LABEL
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

EXPECTED = {
    "phase4d_experiment_sha256": "70ac547c7a257d978cbc241f049fb2dbbc912b0d5801ca28080820d1e183d4c5",
    "phase4d_selected_checkpoint_sha256": "314d3a45009a51159bb74bebed9806af76d1df103a14eca9a38160d0dab0d972",
    "phase4d_nonfinal_report_sha256": "9cc328dd3deeafb175aabaf18ba2daaf9914863b95879368a61416582c96680b",
    "policy_development_predictions_sha256": "401aaf4ee9139fa178155c0fbb1a4efa232d8c43ac69604a79b931d09792d90c",
    "calibration_gate_predictions_sha256": "fa87dde72ea7e268d6a5f80802d1e70b27818346cf612c178ac7cc0402ab3a3a",
    "qasper_threshold": 0.21089398860931396,
    "policy_development_counts": {"fn": 29, "fp": 41, "tp": 10},
    "calibration_gate_counts": {"fn": 62, "fp": 93, "tp": 28},
    "previous_audit_experiment_sha256": "055683587cc016e33cfd889795d5d70f9e16623acf9c75bcb3b5db9ae276ee63",
    "previous_completed_review_sha256": "11e6444c5c8e648eb5052cabe650bd91a66cc1740442ce0e94efc881fdd83e9f",
    "previous_analysis_sha256": "03cd437eb10c6a8894dc072e6f3f3003e9354507b542c0360a5012968fd20043",
}

print(json.dumps(CFG, indent=2, allow_nan=False))
print("Immutable corpus input:", BASE_RUN)
print("Immutable Phase 4D input:", PHASE4D_RUN)
print("Superseded Phase 4E audit:", PREVIOUS_AUDIT_DIR)
print("Phase 4E-A2 output:", AUDIT_DIR)


## 4. Strict JSON, hashing, and atomic-write helpers


In [ ]:
import numpy as np
import pandas as pd

def json_safe(value):
    if isinstance(value, np.ndarray):
        return [json_safe(item) for item in value.tolist()]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if value is pd.NA:
        return None
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value

def canonical_bytes(obj):
    return json.dumps(
        json_safe(obj), sort_keys=True, separators=(",", ":"),
        ensure_ascii=False, allow_nan=False,
    ).encode("utf-8")

def sha256_obj(obj):
    return hashlib.sha256(canonical_bytes(obj)).hexdigest()

def sha256_text(text):
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()

def sha256_file(path, chunk=4 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            h.update(block)
    return h.hexdigest()

def reject_json_constant(token):
    raise ValueError(f"Non-standard JSON numeric constant: {token}")

def read_json(path):
    return json.loads(Path(path).read_text(), parse_constant=reject_json_constant)

def atomic_json(obj, destination):
    destination = Path(destination)
    tmp = destination.with_name(destination.name + ".partial")
    tmp.write_text(json.dumps(
        json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False,
    ))
    os.replace(tmp, destination)

def atomic_text(text, destination):
    destination = Path(destination)
    tmp = destination.with_name(destination.name + ".partial")
    tmp.write_text(text)
    os.replace(tmp, destination)

probe = json.dumps(json_safe({"nan": np.nan, "inf": np.inf}), allow_nan=False)
assert json.loads(probe) == {"nan": None, "inf": None}
try:
    json.loads('{"bad": NaN}', parse_constant=reject_json_constant)
except ValueError:
    pass
else:
    raise AssertionError("Strict JSON reader accepted NaN.")
print("Strict JSON contract passed: unavailable/non-finite values serialize as null.")


## 5. Verify the frozen corpus, Phase 4D result, previous audit, and final boundary


In [ ]:
required_parent = [
    "EXPERIMENT_CONTRACT.json",
    "TRAINING_REPORT.json",
    "NONFINAL_EVALUATION.json",
    "NONFINAL_RESULT_LOCK.json",
    "policy_development_predictions.parquet",
    "calibration_gate_predictions.parquet",
]
missing = [name for name in required_parent if not (PHASE4D_RUN / name).is_file()]
if missing:
    raise FileNotFoundError(f"Incomplete Phase 4D run: {missing}")
if (PHASE4D_RUN / "FINAL_EVALUATION.json").exists():
    raise RuntimeError("Phase 4D final output exists; audit contract requires final closed.")

PARENT_CONTRACT = read_json(PHASE4D_RUN / "EXPERIMENT_CONTRACT.json")
PARENT_TRAINING = read_json(PHASE4D_RUN / "TRAINING_REPORT.json")
PARENT_REPORT = read_json(PHASE4D_RUN / "NONFINAL_EVALUATION.json")
PARENT_LOCK = read_json(PHASE4D_RUN / "NONFINAL_RESULT_LOCK.json")

if PARENT_CONTRACT["experiment_sha256"] != EXPECTED["phase4d_experiment_sha256"]:
    raise RuntimeError("Phase 4D experiment identity changed.")
for record in (PARENT_TRAINING, PARENT_REPORT, PARENT_LOCK):
    if record.get("experiment_sha256") != EXPECTED["phase4d_experiment_sha256"]:
        raise RuntimeError("Phase 4D artifact/contract mismatch.")
    if record.get("final_opened") is not False:
        raise RuntimeError("A Phase 4D artifact does not attest that final stayed closed.")
if PARENT_LOCK["status"] != "nonfinal_failed_final_unavailable":
    raise RuntimeError("Unexpected Phase 4D terminal status.")
if PARENT_LOCK["overall_nonfinal_pass"] is not False:
    raise RuntimeError("Phase 4D lock unexpectedly reports a pass.")
if PARENT_LOCK["selected_checkpoint_sha256"] != EXPECTED["phase4d_selected_checkpoint_sha256"]:
    raise RuntimeError("Phase 4D selected checkpoint identity changed.")
checkpoint = PHASE4D_RUN / PARENT_LOCK["selected_checkpoint"]
if not checkpoint.is_file() or sha256_file(checkpoint) != EXPECTED["phase4d_selected_checkpoint_sha256"]:
    raise RuntimeError("Phase 4D selected checkpoint file changed.")
if sha256_file(PHASE4D_RUN / "NONFINAL_EVALUATION.json") != EXPECTED["phase4d_nonfinal_report_sha256"]:
    raise RuntimeError("Phase 4D non-final report file changed.")

prediction_paths = {
    "policy_development": PHASE4D_RUN / "policy_development_predictions.parquet",
    "calibration_gate": PHASE4D_RUN / "calibration_gate_predictions.parquet",
}
for split, path in prediction_paths.items():
    expected_hash = EXPECTED[f"{split}_predictions_sha256"]
    if sha256_file(path) != expected_hash:
        raise RuntimeError(f"Changed Phase 4D prediction table: {split}")
    if PARENT_REPORT["prediction_files_sha256"][split] != expected_hash:
        raise RuntimeError(f"Phase 4D report hash mismatch: {split}")

threshold_record = PARENT_REPORT[
    "applicability_thresholds_selected_on_policy_development"
]["qasper"]
QASPER_THRESHOLD = float(threshold_record["threshold"])
if not math.isclose(QASPER_THRESHOLD, EXPECTED["qasper_threshold"], rel_tol=0, abs_tol=1e-15):
    raise RuntimeError("Frozen QASPER operating threshold changed.")

previous_required = ["AUDIT_CONTRACT.json", "AUDIT_ANALYSIS.json", "AUDIT_RESULT_LOCK.json"]
missing_previous = [name for name in previous_required if not (PREVIOUS_AUDIT_DIR / name).is_file()]
if missing_previous:
    raise FileNotFoundError(f"Missing superseded Phase 4E artifacts: {missing_previous}")
PREVIOUS_CONTRACT = read_json(PREVIOUS_AUDIT_DIR / "AUDIT_CONTRACT.json")
PREVIOUS_ANALYSIS = read_json(PREVIOUS_AUDIT_DIR / "AUDIT_ANALYSIS.json")
PREVIOUS_LOCK = read_json(PREVIOUS_AUDIT_DIR / "AUDIT_RESULT_LOCK.json")
if PREVIOUS_CONTRACT.get("audit_experiment_sha256") != EXPECTED["previous_audit_experiment_sha256"]:
    raise RuntimeError("Superseded Phase 4E experiment identity changed.")
if PREVIOUS_LOCK.get("completed_review_sha256") != EXPECTED["previous_completed_review_sha256"]:
    raise RuntimeError("Superseded Phase 4E completed-review identity changed.")
if PREVIOUS_LOCK.get("analysis_sha256") != EXPECTED["previous_analysis_sha256"]:
    raise RuntimeError("Superseded Phase 4E analysis identity changed.")
if PREVIOUS_LOCK.get("final_opened") is not False:
    raise RuntimeError("Superseded audit does not attest that final stayed closed.")

CORPUS_ROOT = BASE_RUN / "opendecision-mq-v1"
CORPUS_MANIFEST = read_json(CORPUS_ROOT / "manifest.json")
CORPUS_LOCK = read_json(CORPUS_ROOT / "corpus_lock.json")
if CORPUS_MANIFEST["run_id"] != BASE_RUN_ID:
    raise RuntimeError("Corpus run ID changed.")
if sha256_obj(CORPUS_MANIFEST) != CORPUS_LOCK["manifest_sha256"]:
    raise RuntimeError("Corpus manifest no longer matches its lock.")
if CORPUS_MANIFEST.get("final_labels_or_predictions_opened") is not False:
    raise RuntimeError("Corpus manifest does not attest that final stayed closed.")

frames = {}
for name, expected_hash in CORPUS_LOCK["tables"].items():
    path = CORPUS_ROOT / f"{name}.parquet"
    if sha256_file(path) != expected_hash:
        raise RuntimeError(f"Changed corpus table: {name}")
    frames[name] = pd.read_parquet(path)
    if len(frames[name]) != CORPUS_MANIFEST["counts"][name]:
        raise RuntimeError(f"Changed corpus row count: {name}")

required_state_columns = {
    "state_id", "source", "split", "text", "text_sha256",
}
missing_state_columns = required_state_columns - set(frames["states"].columns)
if missing_state_columns:
    raise RuntimeError(f"Frozen states schema lacks required columns: {sorted(missing_state_columns)}")
state_text = frames["states"]["text"].fillna("").astype(str)
if state_text.str.strip().eq("").any():
    bad = frames["states"].loc[state_text.str.strip().eq(""), "state_id"].head().tolist()
    raise RuntimeError(f"Frozen corpus contains empty state text: {bad}")
hash_matches = [
    sha256_text(text) == expected
    for text, expected in zip(state_text, frames["states"]["text_sha256"])
]
if not all(hash_matches):
    raise RuntimeError("Frozen corpus state text does not match text_sha256.")

final_state_ids = set(frames["states"].loc[
    frames["states"]["split"] == "final", "state_id"
])
if not final_state_ids:
    raise RuntimeError("Expected a reserved final-state set.")

INPUT_LOCK = {
    "phase4d_experiment_sha256": PARENT_CONTRACT["experiment_sha256"],
    "phase4d_checkpoint_sha256": PARENT_LOCK["selected_checkpoint_sha256"],
    "phase4d_nonfinal_report_sha256": sha256_file(PHASE4D_RUN / "NONFINAL_EVALUATION.json"),
    "prediction_files_sha256": {
        split: sha256_file(path) for split, path in prediction_paths.items()
    },
    "corpus_manifest_sha256": CORPUS_LOCK["manifest_sha256"],
    "corpus_tables_sha256": CORPUS_LOCK["tables"],
    "qasper_threshold": QASPER_THRESHOLD,
    "previous_audit_experiment_sha256": PREVIOUS_CONTRACT["audit_experiment_sha256"],
    "previous_completed_review_sha256": PREVIOUS_LOCK["completed_review_sha256"],
    "previous_analysis_sha256": PREVIOUS_LOCK["analysis_sha256"],
    "state_text_column": "text",
    "final_opened": False,
}
print(json.dumps(INPUT_LOCK, indent=2, allow_nan=False))


## 6. Reconstruct frozen-threshold QASPER outcomes


In [ ]:
def parse_list(value):
    if isinstance(value, str):
        return json.loads(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return list(value)
    raise TypeError(f"Expected a serialized list, got {type(value).__name__}")

def add_operating_fields(frame, split):
    out = frame.loc[frame["source"].str.lower() == "qasper"].copy()
    out["partition"] = split

    def none_probability(row):
        option_ids = parse_list(row["option_ids"])
        probabilities = [float(x) for x in parse_list(row["probabilities"])]
        if len(option_ids) != len(probabilities) or "__none__" not in option_ids:
            raise RuntimeError(f"Invalid option/probability row: {row['question_id']}")
        if not math.isclose(sum(probabilities), 1.0, rel_tol=0, abs_tol=2e-5):
            raise RuntimeError(f"Probability sum error: {row['question_id']}")
        return probabilities[option_ids.index("__none__")]

    out["none_probability"] = out.apply(none_probability, axis=1)
    out["locked_semantic_none"] = out["target_label"].eq("__none__")
    out["operating_predicted_none"] = out["none_probability"].ge(QASPER_THRESHOLD)
    conditions = [
        out["locked_semantic_none"] & out["operating_predicted_none"],
        out["locked_semantic_none"] & ~out["operating_predicted_none"],
        ~out["locked_semantic_none"] & out["operating_predicted_none"],
    ]
    out["operating_outcome"] = np.select(conditions, ["tp", "fn", "fp"], default="tn")
    if out["question_id"].duplicated().any():
        raise RuntimeError(f"Duplicate QASPER question IDs in {split} predictions.")
    return out

prediction_frames = {
    split: add_operating_fields(pd.read_parquet(path), split)
    for split, path in prediction_paths.items()
}
policy = prediction_frames["policy_development"]
gate = prediction_frames["calibration_gate"]

def outcome_counts(frame):
    counts = frame["operating_outcome"].value_counts().to_dict()
    return {key: int(counts.get(key, 0)) for key in ("tp", "fn", "fp", "tn")}

COUNTS = {split: outcome_counts(frame) for split, frame in prediction_frames.items()}
for split in ("policy_development", "calibration_gate"):
    for key, expected_count in EXPECTED[f"{split}_counts"].items():
        if COUNTS[split][key] != expected_count:
            raise RuntimeError(
                f"Frozen {split} {key} count changed: "
                f"{COUNTS[split][key]} != {expected_count}"
            )
assert len(policy) == 320 and len(gate) == 692
assert COUNTS["policy_development"]["fn"] + COUNTS["policy_development"]["fp"] == 70

count_table = pd.DataFrame(COUNTS).T.reset_index(names="partition")
display(count_table)
print("Frozen QASPER threshold:", QASPER_THRESHOLD)


## 7. Preserve Phase 4A–4D context without starting another sweep


In [ ]:
import matplotlib.pyplot as plt

phase_summary = pd.DataFrame([
    {"experiment": "cap-12 scalar", "qasper_recall": 0.1556, "gate_policy_cost": 0.10384},
    {"experiment": "stratified parent", "qasper_recall": 0.3111, "gate_policy_cost": 0.11675},
    {"experiment": "pairwise", "qasper_recall": 0.3333, "gate_policy_cost": 0.10221},
])
display(phase_summary)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(phase_summary["experiment"], phase_summary["qasper_recall"], color="#3B82F6")
axes[0].axhline(0.30, color="black", linestyle="--", label="recall floor")
axes[0].set_title("QASPER calibration-gate none recall")
axes[0].set_ylabel("Recall")
axes[0].tick_params(axis="x", rotation=20)
axes[0].legend()

for target, group in policy.groupby("locked_semantic_none"):
    axes[1].hist(
        group["none_probability"], bins=24, alpha=0.55,
        label="semantic none" if target else "answerable",
    )
axes[1].axvline(QASPER_THRESHOLD, color="black", linestyle="--", label="frozen threshold")
axes[1].set_title("Policy-development QASPER none scores")
axes[1].set_xlabel("None probability")
axes[1].set_ylabel("Questions")
axes[1].legend()
plt.tight_layout()
plt.show()


## 8. Rebuild the deterministic 150-row audit population


In [ ]:
def stable_digest(*parts):
    payload = "|".join(str(part) for part in parts).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def state_round_robin_sample(frame, n, salt):
    if n > len(frame):
        raise ValueError(f"Requested {n} rows from only {len(frame)} candidates.")
    work = frame.copy()
    work["_state_order"] = work["state_id"].map(
        lambda value: stable_digest(SEED, salt, "state", value)
    )
    work["_item_order"] = work.apply(
        lambda row: stable_digest(SEED, salt, "item", row["state_id"], row["question_id"]),
        axis=1,
    )
    grouped = {
        state_id: list(group.sort_values("_item_order").index)
        for state_id, group in work.groupby("state_id", sort=False)
    }
    state_order = sorted(grouped, key=lambda state_id: stable_digest(SEED, salt, state_id))
    selected = []
    while len(selected) < n:
        added = False
        for state_id in state_order:
            if grouped[state_id] and len(selected) < n:
                selected.append(grouped[state_id].pop(0))
                added = True
        if not added:
            break
    if len(selected) != n:
        raise RuntimeError("Deterministic state-round-robin sampling underfilled.")
    return work.loc[selected].drop(columns=["_state_order", "_item_order"])

selected_parts = []
policy_errors = policy.loc[policy["operating_outcome"].isin(["fn", "fp"])].copy()
policy_errors["selection_role"] = "policy_all_error"
selected_parts.append(policy_errors)

for outcome, n in (("tp", 10), ("tn", 10)):
    sample = state_round_robin_sample(
        policy.loc[policy["operating_outcome"] == outcome], n,
        f"policy_control_{outcome}",
    )
    sample["selection_role"] = "policy_blinded_control"
    selected_parts.append(sample)

for outcome, n in (("fn", 30), ("fp", 30)):
    sample = state_round_robin_sample(
        gate.loc[gate["operating_outcome"] == outcome], n,
        f"gate_error_{outcome}",
    )
    sample["selection_role"] = "gate_state_diverse_error_sample"
    selected_parts.append(sample)

AUDIT_KEY_FRAME = pd.concat(selected_parts, ignore_index=True)
if len(AUDIT_KEY_FRAME) != 150:
    raise AssertionError(f"Expected 150 audit items, got {len(AUDIT_KEY_FRAME)}")
if AUDIT_KEY_FRAME["question_id"].duplicated().any():
    raise RuntimeError("Audit population contains duplicate question IDs.")
if set(AUDIT_KEY_FRAME["state_id"]) & final_state_ids:
    raise RuntimeError("Audit population includes a final state.")

AUDIT_KEY_FRAME["review_id"] = AUDIT_KEY_FRAME.apply(
    lambda row: "ok4e2_" + stable_digest(
        "openkind-phase4e-a2", row["partition"], row["question_id"]
    )[:16], axis=1,
)
AUDIT_KEY_FRAME["blinded_order"] = AUDIT_KEY_FRAME["review_id"].map(
    lambda value: stable_digest(SEED, "phase4e_a2_blinded_order", value)
)
AUDIT_KEY_FRAME = AUDIT_KEY_FRAME.sort_values("blinded_order").reset_index(drop=True)

selection_summary = (
    AUDIT_KEY_FRAME.groupby(["partition", "selection_role", "operating_outcome"])
    .size().rename("n").reset_index()
)
display(selection_summary)


## 9. Join canonical paper text and keep gold evidence locked


In [ ]:
states = frames["states"].set_index("state_id", drop=False)
questions = frames["questions"].set_index("question_id", drop=False)

option_groups = {}
for question_id, group in frames["options"].groupby("question_id"):
    if "position" in group.columns:
        group = group.sort_values("position")
    option_groups[question_id] = [
        {"option_id": str(row["option_id"]), "description": str(row.get("description", ""))}
        for _, row in group.iterrows()
    ]

evidence_groups = {}
if "evidence" in frames:
    for question_id, group in frames["evidence"].groupby("question_id"):
        values = []
        for value in group.get("text", pd.Series(dtype=object)).tolist():
            if pd.notna(value) and str(value).strip() and str(value) not in values:
                values.append(str(value))
        evidence_groups[question_id] = values

review_rows = []
key_rows = []
evidence_rows = []
for _, model_row in AUDIT_KEY_FRAME.iterrows():
    state_id = model_row["state_id"]
    question_id = model_row["question_id"]
    if state_id not in states.index or question_id not in questions.index:
        raise RuntimeError(f"Missing corpus record for {question_id}")
    state_row = states.loc[state_id]
    question_row = questions.loc[question_id]
    if str(state_row["split"]) != str(model_row["partition"]):
        raise RuntimeError(f"Prediction/corpus split mismatch for {question_id}")
    if str(state_row["source"]).lower() != "qasper":
        raise RuntimeError(f"Non-QASPER state selected: {state_id}")

    # Canonical field: fail loudly instead of silently falling back to an empty string.
    state_text_value = state_row["text"]
    if pd.isna(state_text_value) or not str(state_text_value).strip():
        raise RuntimeError(f"Empty canonical state text for {state_id}")
    state_text_value = str(state_text_value)
    if sha256_text(state_text_value) != str(state_row["text_sha256"]):
        raise RuntimeError(f"State text hash mismatch for {state_id}")

    options_text = "\n".join(
        f"- {item['option_id']}: {item['description']}"
        for item in option_groups.get(question_id, [])
    )
    evidence_values = evidence_groups.get(question_id, [])
    evidence_text = "\n\n".join(evidence_values)
    evidence_sha256 = sha256_text(evidence_text)

    review_rows.append({
        "review_id": model_row["review_id"],
        "partition": model_row["partition"],
        "source": "qasper",
        "state_id": state_id,
        "question_id": question_id,
        "state_text_sha256": str(state_row["text_sha256"]),
        "state_text_chars": len(state_text_value),
        "state_text": state_text_value,
        "instruction": str(question_row.get("instruction", "")),
        "options": options_text,
        "review_target_kind": "",
        "review_reason_code": "",
        "review_confidence": "",
        "review_notes": "",
        "reviewer_id": "",
    })
    key_rows.append({
        "review_id": model_row["review_id"],
        "partition": model_row["partition"],
        "state_id": state_id,
        "question_id": question_id,
        "selection_role": model_row["selection_role"],
        "locked_target_kind": "semantic_none" if model_row["locked_semantic_none"] else "answerable",
        "target_label": model_row["target_label"],
        "model_prediction_label": model_row["prediction_label"],
        "none_probability": float(model_row["none_probability"]),
        "operating_threshold": QASPER_THRESHOLD,
        "operating_predicted_none": bool(model_row["operating_predicted_none"]),
        "operating_outcome": model_row["operating_outcome"],
        "evidence_count": len(evidence_values),
        "evidence_present": bool(evidence_values),
        "evidence_sha256": evidence_sha256,
    })
    evidence_rows.append({
        "review_id": model_row["review_id"],
        "question_id": question_id,
        "evidence_count": len(evidence_values),
        "evidence_sha256": evidence_sha256,
        "evidence": evidence_text,
    })

REVIEW_FRAME = pd.DataFrame(review_rows)
KEY_FRAME = pd.DataFrame(key_rows)
EVIDENCE_LOCKED_FRAME = pd.DataFrame(evidence_rows)
if REVIEW_FRAME["review_id"].tolist() != KEY_FRAME["review_id"].tolist():
    raise RuntimeError("Blinded review and key order diverged.")
if REVIEW_FRAME["review_id"].tolist() != EVIDENCE_LOCKED_FRAME["review_id"].tolist():
    raise RuntimeError("Blinded review and evidence-lock order diverged.")

forbidden = {
    "locked_target_kind", "target_label", "model_prediction_label",
    "none_probability", "operating_threshold", "operating_predicted_none",
    "operating_outcome", "selection_role", "evidence", "evidence_count",
    "evidence_present", "evidence_sha256",
}
if forbidden & set(REVIEW_FRAME.columns):
    raise RuntimeError("Reviewer-visible table leaked a locked field.")
display(REVIEW_FRAME[[
    "review_id", "partition", "state_id", "question_id",
    "state_text_chars", "instruction",
]].head(3))


## 10. Run the audit-pack quality gate


In [ ]:
review_text = REVIEW_FRAME["state_text"].fillna("").astype(str)
hash_ok = [
    sha256_text(text) == expected
    for text, expected in zip(review_text, REVIEW_FRAME["state_text_sha256"])
]
PACK_QA = {
    "schema": "openkind-phase4e-a2-audit-packet-qa/v1",
    "rows": int(len(REVIEW_FRAME)),
    "unique_review_ids": int(REVIEW_FRAME["review_id"].nunique()),
    "unique_question_ids": int(REVIEW_FRAME["question_id"].nunique()),
    "unique_state_ids": int(REVIEW_FRAME["state_id"].nunique()),
    "state_text_column": "text",
    "nonempty_state_text_n": int(review_text.str.strip().ne("").sum()),
    "nonempty_state_text_rate": float(review_text.str.strip().ne("").mean()),
    "state_text_hash_match_n": int(sum(hash_ok)),
    "state_text_hash_match_rate": float(np.mean(hash_ok)),
    "state_text_chars_min": int(REVIEW_FRAME["state_text_chars"].min()),
    "state_text_chars_median": float(REVIEW_FRAME["state_text_chars"].median()),
    "state_text_chars_max": int(REVIEW_FRAME["state_text_chars"].max()),
    "gold_evidence_visible_to_primary_reviewer": False,
    "locked_or_model_fields_visible_to_primary_reviewer": False,
    "final_opened": False,
}

required_qa = {
    "rows": 150,
    "unique_review_ids": 150,
    "unique_question_ids": 150,
    "nonempty_state_text_n": 150,
    "nonempty_state_text_rate": 1.0,
    "state_text_hash_match_n": 150,
    "state_text_hash_match_rate": 1.0,
}
for field, expected_value in required_qa.items():
    if PACK_QA[field] != expected_value:
        raise RuntimeError(f"Audit-pack QA failed: {field}={PACK_QA[field]!r}, expected {expected_value!r}")

print(json.dumps(PACK_QA, indent=2, allow_nan=False))
length_summary = REVIEW_FRAME["state_text_chars"].describe(
    percentiles=[0.25, 0.5, 0.75, 0.95]
).to_frame("characters")
display(length_summary)


## 11. Write or verify the repaired audit contract and artifacts


In [ ]:
REVIEW_PATH = AUDIT_DIR / "AUDIT_REVIEW_BLINDED.csv"
COMPLETED_REVIEW_PATH = AUDIT_DIR / "AUDIT_REVIEW_COMPLETED.csv"
KEY_PATH = AUDIT_DIR / "AUDIT_KEY.parquet"
EVIDENCE_LOCKED_PATH = AUDIT_DIR / "AUDIT_EVIDENCE_LOCKED.parquet"
GUIDE_PATH = AUDIT_DIR / "AUDIT_GUIDE.md"
CONTRACT_PATH = AUDIT_DIR / "AUDIT_CONTRACT.json"
MANIFEST_PATH = AUDIT_DIR / "AUDIT_MANIFEST.json"
PACK_QA_PATH = AUDIT_DIR / "AUDIT_PACKET_QA.json"

repair_provenance = {
    "superseded_audit_experiment_sha256": PREVIOUS_CONTRACT["audit_experiment_sha256"],
    "superseded_completed_review_sha256": PREVIOUS_LOCK["completed_review_sha256"],
    "superseded_analysis_sha256": PREVIOUS_LOCK["analysis_sha256"],
    "superseded_overall": PREVIOUS_ANALYSIS["overall"],
    "defect": "all reviewer-visible state_text values were empty",
    "root_cause": "nonexistent effective_state_text field was accessed with a silent empty-string fallback",
    "repair": "use canonical states.text; verify text_sha256; require 100% non-empty coverage; hide gold evidence",
}

contract_payload = {
    "schema": "openkind-phase4e-a2-qasper-audit-repair-contract/v1",
    "config": {**CFG, "mode": "prepare_audit_or_analyze_review"},
    "input_lock": INPUT_LOCK,
    "repair_provenance": repair_provenance,
    "selection_counts": selection_summary.to_dict(orient="records"),
    "packet_quality_gate": required_qa,
    "blinding": {
        "reviewer_visible": list(REVIEW_FRAME.columns),
        "key_only": list(KEY_FRAME.columns),
        "evidence_locked": list(EVIDENCE_LOCKED_FRAME.columns),
        "forbidden_from_primary_review": sorted(forbidden),
    },
    "automatic_phase4e_b_authorization": False,
    "final_opened": False,
}
AUDIT_EXPERIMENT_SHA256 = sha256_obj(contract_payload)
AUDIT_CONTRACT = {**contract_payload, "audit_experiment_sha256": AUDIT_EXPERIMENT_SHA256}

guide = f"""# OpenKind Phase 4E-A2 QASPER Repaired Audit Guide

Audit experiment: `{AUDIT_EXPERIMENT_SHA256}`

Review `AUDIT_REVIEW_BLINDED.csv` without opening `AUDIT_KEY.parquet` or `AUDIT_EVIDENCE_LOCKED.parquet`.

For each row, decide whether the **paper text in `state_text`** contains enough information to answer `instruction`:

- `answerable`: the state text explicitly or implicitly supports an answer;
- `semantic_none`: the state text does not contain enough information to answer; or
- `ambiguous`: the question/state is genuinely underspecified, contradictory, or otherwise cannot be classified after reading the state.

Complete every reviewer field:

- `review_target_kind`;
- `review_reason_code` (recommended: `explicit_answer`, `implicit_answer`, `missing_evidence`, `question_underspecified`, `conflicting_evidence`, `other`);
- `review_confidence` from 0 through 1;
- `review_notes`; and
- `reviewer_id`.

Save the completed file as `AUDIT_REVIEW_COMPLETED.csv` in this directory. Do not reorder, delete, duplicate, or add rows; do not edit IDs, hashes, lengths, or source text. Do not use gold evidence, model predictions, none scores, error types, locked labels, or the superseded review during the primary review.

After the primary analysis, inspect only the rows emitted to `AUDIT_ADJUDICATION_PACKET.csv`. Phase 4E-B is never automatically authorized. Final remains closed.
"""

def prepared_artifact_hashes():
    names = [
        REVIEW_PATH.name, KEY_PATH.name, EVIDENCE_LOCKED_PATH.name,
        GUIDE_PATH.name, CONTRACT_PATH.name, PACK_QA_PATH.name,
    ]
    return {name: sha256_file(AUDIT_DIR / name) for name in names}

if MODE == "prepare_audit":
    core_paths = [REVIEW_PATH, KEY_PATH, EVIDENCE_LOCKED_PATH, GUIDE_PATH, CONTRACT_PATH, PACK_QA_PATH, MANIFEST_PATH]
    existing = [path.exists() for path in core_paths]
    if any(existing) and not all(existing):
        raise RuntimeError("Partial repaired audit pack exists; refusing to overwrite it.")

    if all(existing):
        saved_contract = read_json(CONTRACT_PATH)
        saved_manifest = read_json(MANIFEST_PATH)
        if saved_contract.get("audit_experiment_sha256") != AUDIT_EXPERIMENT_SHA256:
            raise RuntimeError("Existing repaired audit contract differs from this notebook.")
        for name, expected_hash in saved_manifest["artifacts_sha256"].items():
            if sha256_file(AUDIT_DIR / name) != expected_hash:
                raise RuntimeError(f"Existing repaired audit artifact changed: {name}")
        if read_json(PACK_QA_PATH) != PACK_QA:
            raise RuntimeError("Existing audit packet QA record differs from recomputed QA.")
        print("Existing repaired audit pack verified; no files were overwritten:", AUDIT_DIR)
    else:
        review_tmp = REVIEW_PATH.with_name(REVIEW_PATH.name + ".partial")
        REVIEW_FRAME.to_csv(review_tmp, index=False, lineterminator="\n")
        os.replace(review_tmp, REVIEW_PATH)

        key_tmp = KEY_PATH.with_name(KEY_PATH.name + ".partial")
        KEY_FRAME.to_parquet(key_tmp, index=False)
        os.replace(key_tmp, KEY_PATH)

        evidence_tmp = EVIDENCE_LOCKED_PATH.with_name(EVIDENCE_LOCKED_PATH.name + ".partial")
        EVIDENCE_LOCKED_FRAME.to_parquet(evidence_tmp, index=False)
        os.replace(evidence_tmp, EVIDENCE_LOCKED_PATH)

        atomic_text(guide.strip() + "\n", GUIDE_PATH)
        atomic_json(AUDIT_CONTRACT, CONTRACT_PATH)
        atomic_json(PACK_QA, PACK_QA_PATH)

        manifest_payload = {
            "schema": "openkind-phase4e-a2-qasper-audit-repair-manifest/v1",
            "audit_experiment_sha256": AUDIT_EXPERIMENT_SHA256,
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "rows": len(REVIEW_FRAME),
            "unique_states": int(REVIEW_FRAME["state_id"].nunique()),
            "unique_questions": int(REVIEW_FRAME["question_id"].nunique()),
            "selection_counts": selection_summary.to_dict(orient="records"),
            "source_outcome_counts": COUNTS,
            "packet_qa": PACK_QA,
            "artifacts_sha256": prepared_artifact_hashes(),
            "review_completed": False,
            "phase4e_b_authorized": False,
            "final_opened": False,
        }
        atomic_json(manifest_payload, MANIFEST_PATH)
        print("Prepared repaired blinded audit pack:", AUDIT_DIR)
        print(json.dumps(manifest_payload, indent=2, allow_nan=False))
else:
    for path in (REVIEW_PATH, KEY_PATH, EVIDENCE_LOCKED_PATH, GUIDE_PATH, CONTRACT_PATH, PACK_QA_PATH, MANIFEST_PATH):
        if not path.is_file():
            raise FileNotFoundError(f"Missing prepared repaired-audit artifact: {path.name}")
    saved_contract = read_json(CONTRACT_PATH)
    if saved_contract.get("audit_experiment_sha256") != AUDIT_EXPERIMENT_SHA256:
        raise RuntimeError("Prepared repaired-audit contract differs from this notebook configuration.")
    saved_manifest = read_json(MANIFEST_PATH)
    for name, expected_hash in saved_manifest["artifacts_sha256"].items():
        if sha256_file(AUDIT_DIR / name) != expected_hash:
            raise RuntimeError(f"Prepared repaired-audit artifact changed: {name}")
    if read_json(PACK_QA_PATH) != PACK_QA:
        raise RuntimeError("Saved audit packet QA record differs from recomputed QA.")
    print("Prepared repaired-audit artifacts verified for analysis.")


## Human review pause

In `prepare_audit` mode, stop here. Complete the primary blinded review using the full paper text in `state_text`; gold evidence remains locked. Save the result as `AUDIT_REVIEW_COMPLETED.csv` in the same output directory.

Then switch only `MODE` to `"analyze_review"` and rerun. The next cell validates every row and source field before unblinding. It emits a smaller `AUDIT_ADJUDICATION_PACKET.csv` containing only ambiguous rows and disagreements for the post-review decision.


## 12. Validate, unblind, and summarize a completed repaired review


In [ ]:
ANALYSIS = None
if MODE == "prepare_audit":
    print("Repaired audit pack is ready. Complete the primary blinded review before analysis.")
else:
    if not COMPLETED_REVIEW_PATH.is_file():
        raise FileNotFoundError(
            f"Complete the primary blinded review and save it as {COMPLETED_REVIEW_PATH.name}."
        )
    completed = pd.read_csv(COMPLETED_REVIEW_PATH, keep_default_na=False)
    prepared = pd.read_csv(REVIEW_PATH, keep_default_na=False)
    key = pd.read_parquet(KEY_PATH)
    locked_evidence = pd.read_parquet(EVIDENCE_LOCKED_PATH)

    if completed["review_id"].duplicated().any():
        raise ValueError("Completed review contains duplicate review IDs.")
    if set(completed["review_id"]) != set(prepared["review_id"]):
        missing_ids = sorted(set(prepared["review_id"]) - set(completed["review_id"]))
        extra_ids = sorted(set(completed["review_id"]) - set(prepared["review_id"]))
        raise ValueError(f"Completed review ID mismatch; missing={missing_ids[:5]}, extra={extra_ids[:5]}")
    immutable_columns = [
        "review_id", "partition", "source", "state_id", "question_id",
        "state_text_sha256", "state_text_chars", "state_text", "instruction", "options",
    ]
    lhs = completed[immutable_columns].sort_values("review_id").reset_index(drop=True)
    rhs = prepared[immutable_columns].sort_values("review_id").reset_index(drop=True)
    if not lhs.equals(rhs):
        raise ValueError("Completed review changed an immutable ID, hash, length, or source-text field.")

    required_fields = [
        "review_target_kind", "review_reason_code", "review_confidence",
        "review_notes", "reviewer_id",
    ]
    for column in required_fields:
        if completed[column].astype(str).str.strip().eq("").any():
            raise ValueError(f"Completed review has blank values in {column}.")
    allowed = set(CFG["review_target_kinds"])
    observed = set(completed["review_target_kind"].str.strip())
    if not observed <= allowed:
        raise ValueError(f"Invalid review_target_kind values: {sorted(observed - allowed)}")
    completed["review_confidence"] = pd.to_numeric(completed["review_confidence"], errors="raise")
    if not completed["review_confidence"].between(0, 1).all():
        raise ValueError("review_confidence must be between 0 and 1.")

    joined = completed.merge(
        key, on=["review_id", "partition", "state_id", "question_id"], validate="one_to_one"
    ).merge(
        locked_evidence, on=["review_id", "question_id"], validate="one_to_one"
    )
    if not (joined["evidence_sha256_x"] == joined["evidence_sha256_y"]).all():
        raise RuntimeError("Locked key and evidence packet hashes diverged.")
    joined = joined.rename(columns={
        "evidence_count_x": "evidence_count",
        "evidence_sha256_x": "evidence_sha256",
    }).drop(columns=["evidence_count_y", "evidence_sha256_y"])
    joined["is_ambiguous"] = joined["review_target_kind"].eq("ambiguous")
    joined["agrees_with_locked"] = np.where(
        joined["is_ambiguous"], pd.NA,
        joined["review_target_kind"].eq(joined["locked_target_kind"]),
    )

    def summarize(group):
        decided = group.loc[~group["is_ambiguous"]]
        return pd.Series({
            "n": len(group),
            "ambiguous_n": int(group["is_ambiguous"].sum()),
            "ambiguous_rate": float(group["is_ambiguous"].mean()),
            "decided_n": len(decided),
            "decided_coverage": float(len(decided) / len(group)) if len(group) else None,
            "locked_label_agreement": (
                None if decided.empty else float(decided["agrees_with_locked"].astype(bool).mean())
            ),
            "mean_review_confidence": float(group["review_confidence"].mean()),
        })

    by_partition = joined.groupby("partition", dropna=False).apply(summarize, include_groups=False).reset_index()
    by_outcome = joined.groupby("operating_outcome", dropna=False).apply(summarize, include_groups=False).reset_index()
    by_role = joined.groupby("selection_role", dropna=False).apply(summarize, include_groups=False).reset_index()
    by_locked_target = joined.groupby("locked_target_kind", dropna=False).apply(summarize, include_groups=False).reset_index()
    by_evidence_presence = joined.groupby("evidence_present", dropna=False).apply(summarize, include_groups=False).reset_index()
    overall = summarize(joined).to_dict()

    adjudication = joined.loc[
        joined["is_ambiguous"] | joined["agrees_with_locked"].eq(False),
        [
            "review_id", "partition", "selection_role", "operating_outcome",
            "state_id", "question_id", "instruction", "options", "state_text",
            "review_target_kind", "review_reason_code", "review_confidence", "review_notes",
            "locked_target_kind", "target_label", "model_prediction_label",
            "none_probability", "operating_threshold", "evidence_count", "evidence",
        ],
    ].copy()
    adjudication["adjudicated_target_kind"] = ""
    adjudication["adjudication_reason_code"] = ""
    adjudication["adjudication_notes"] = ""
    adjudication["adjudicator_id"] = ""
    adjudication_path = AUDIT_DIR / "AUDIT_ADJUDICATION_PACKET.csv"
    adjudication.to_csv(adjudication_path, index=False, lineterminator="\n")

    ANALYSIS = {
        "schema": "openkind-phase4e-a2-qasper-audit-repair-analysis/v1",
        "audit_experiment_sha256": AUDIT_EXPERIMENT_SHA256,
        "completed_review_sha256": sha256_file(COMPLETED_REVIEW_PATH),
        "analyzed_at_utc": datetime.now(timezone.utc).isoformat(),
        "packet_qa": PACK_QA,
        "overall": overall,
        "by_partition": by_partition.to_dict(orient="records"),
        "by_operating_outcome": by_outcome.to_dict(orient="records"),
        "by_selection_role": by_role.to_dict(orient="records"),
        "by_locked_target_kind": by_locked_target.to_dict(orient="records"),
        "by_evidence_presence": by_evidence_presence.to_dict(orient="records"),
        "reason_counts": (
            joined["review_reason_code"].value_counts().rename_axis("reason").reset_index(name="n").to_dict(orient="records")
        ),
        "reviewer_counts": (
            joined["reviewer_id"].value_counts().rename_axis("reviewer_id").reset_index(name="n").to_dict(orient="records")
        ),
        "adjudication_rows": int(len(adjudication)),
        "adjudication_packet_sha256": sha256_file(adjudication_path),
        "automatic_phase4e_b_authorization": False,
        "required_next_decision": (
            "Complete the emitted adjudication packet. If semantic-none labels are frequently overturned, "
            "repair data; if they are confirmed, preregister one representation-aware seed-17 arm."
        ),
        "final_opened": False,
    }
    analysis_path = AUDIT_DIR / "AUDIT_ANALYSIS.json"
    atomic_json(ANALYSIS, analysis_path)
    result_lock = {
        "schema": "openkind-phase4e-a2-qasper-audit-repair-result-lock/v1",
        "audit_experiment_sha256": AUDIT_EXPERIMENT_SHA256,
        "completed_review_sha256": sha256_file(COMPLETED_REVIEW_PATH),
        "analysis_sha256": sha256_file(analysis_path),
        "adjudication_packet_sha256": sha256_file(adjudication_path),
        "review_complete": True,
        "adjudication_complete": False,
        "phase4e_b_automatically_authorized": False,
        "status": "repaired_audit_primary_review_complete_adjudication_required",
        "final_opened": False,
    }
    atomic_json(result_lock, AUDIT_DIR / "AUDIT_RESULT_LOCK.json")

    display(pd.DataFrame([overall]))
    display(by_partition)
    display(by_locked_target)
    display(by_outcome)
    display(by_evidence_presence)
    print(json.dumps(ANALYSIS, indent=2, allow_nan=False))


## 13. Review-result visualization


In [ ]:
if MODE == "analyze_review" and ANALYSIS is not None:
    locked_plot = pd.DataFrame(ANALYSIS["by_locked_target_kind"])
    outcome_plot = pd.DataFrame(ANALYSIS["by_operating_outcome"])
    figure, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    axes[0].bar(locked_plot["locked_target_kind"], locked_plot["decided_coverage"], color="#F59E0B")
    axes[0].set_title("Decided coverage by locked target")
    axes[0].set_ylabel("Coverage")
    axes[0].set_ylim(0, 1)

    axes[1].bar(
        locked_plot["locked_target_kind"],
        locked_plot["locked_label_agreement"].fillna(0),
        color="#10B981",
    )
    axes[1].set_title("Locked-label agreement (decided only)")
    axes[1].set_ylabel("Agreement")
    axes[1].set_ylim(0, 1)

    axes[2].bar(outcome_plot["operating_outcome"], outcome_plot["ambiguous_rate"], color="#8B5CF6")
    axes[2].set_title("Ambiguity by frozen outcome")
    axes[2].set_ylabel("Ambiguous rate")
    axes[2].set_ylim(0, 1)
    plt.tight_layout()
    plt.show()
else:
    print("Review visualization becomes available after analyze_review completes.")


## 14. Terminal guard: Phase 4E-A2 never opens final or starts training


In [ ]:
def open_final_split(*args, **kwargs):
    raise PermissionError(
        "Final evaluation is intentionally unavailable in Phase 4E-A2. "
        "This notebook is a repaired non-final blinded audit, not a promotion run."
    )

if CFG["final_opened"] is not False or INPUT_LOCK["final_opened"] is not False:
    raise AssertionError("Invalid final status in Phase 4E-A2 audit contract.")

if MODE == "prepare_audit":
    print(
        "PHASE 4E-A2 REPAIRED PACK READY. State text coverage and hashes passed; "
        "gold evidence remains locked. Complete AUDIT_REVIEW_COMPLETED.csv, then "
        "switch MODE to analyze_review. Final remains closed."
    )
else:
    if ANALYSIS is None:
        raise AssertionError("Analyze mode did not produce an analysis.")
    print(
        "PHASE 4E-A2 PRIMARY REVIEW ANALYZED. Complete the emitted adjudication packet. "
        "No training is automatically authorized. Final remains closed."
    )
